# Capítulo 10 — Aletas e Superfícies Estendidas

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Calcular** o parâmetro $m = \sqrt{hP/(kA_c)}$ para diferentes geometrias
2. **Determinar** a eficiência de aleta $\eta_{\text{fin}}$ para aletas retangulares, cilíndricas e anulares
3. **Calcular** a efetividade de aleta $\varepsilon_{\text{fin}}$ e interpretar seu significado físico
4. **Estimar** o calor dissipado por uma aleta individual
5. **Dimensionar** um dissipador térmico com múltiplas aletas
6. **Implementar** esquema numérico por Diferenças Finitas para aletas com condições de contorno complexas

---

## 📚 Conteúdo Teórico Resumido

### 10.1 Equação da Aleta

A equação diferencial que governa o perfil de temperatura em uma aleta é:

$$\frac{d^2\theta}{dx^2} - m^2\theta = 0$$

onde $\theta = T - T_\infty$ e $m = \sqrt{hP/(kA_c)}$.

### 10.2 Solução para Ponta Adiabática

$$\theta(x) = \theta_b \frac{\cosh[m(L - x)]}{\cosh(mL)}$$

### 10.3 Calor Dissipado

$$q_{\text{fin}} = \sqrt{hPkA_c} \cdot \theta_b \cdot \tanh(mL)$$

### 10.4 Eficiência de Aleta

$$\eta_{\text{fin}} = \frac{q_{\text{fin}}}{q_{\text{max}}} = \frac{\tanh(mL_c)}{mL_c}$$

### 10.5 Efetividade de Aleta

$$\varepsilon_{\text{fin}} = \frac{q_{\text{fin}}}{hA_b\theta_b}$$

Critério prático: aleta justificável se $\varepsilon_{\text{fin}} > 2$.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
print('✅ Ambiente configurado com sucesso!')

---

## 🌡️ Seção 1: Parâmetro $m$ e Eficiência de Aleta Reta

### Exercício 1.1: Aleta Retangular de Alumínio

Calcule a eficiência $\eta_{\text{fin}}$ para uma aleta retangular de alumínio ($k = 205$ W/(m·K)) com $L = 40$ mm, $t = 3$ mm, $w = 60$ mm, exposta a ar com $h = 50$ W/(m²·K). Use $L_c = L + t/2$.

In [ ]:
# Dados
k = 205.0       # W/(m*K)
L = 0.040       # m
t = 0.003       # m
w = 0.060       # m
h = 50.0        # W/(m^2*K)

# Geometria
Lc = L + t/2
Ac = t * w
P = 2 * (t + w)

# Parâmetro m
m = np.sqrt(h * P / (k * Ac))
mLc = m * Lc

# Eficiência
eta = np.tanh(mLc) / mLc

print('Geometria:')
print('  Lc = {:.1f} mm'.format(Lc*1000))
print('  Ac = {:.2e} m^2'.format(Ac))
print('  P  = {:.3f} m'.format(P))
print('\nParametro m:')
print('  m   = {:.2f} m^-1'.format(m))
print('  mLc = {:.3f}'.format(mLc))
print('\nEficiencia:')
print('  eta = {:.3f} ({:.1f}%)'.format(eta, eta*100))
print('\nA aleta opera com alta eficiencia, aproveitando bem sua area superficial.')

---

## 📍 Seção 2: Aleta Cilíndrica (Pin)

### Exercício 2.1: Pin de Cobre

Uma aleta cilíndrica (pin) de cobre ($k = 400$ W/(m·K)) com $D = 5$ mm e $L = 25$ mm dissipa calor para ar com $h = 80$ W/(m²·K). Calcule a efetividade $\varepsilon_{\text{fin}}$ e verifique se a aleta é justificável ($\varepsilon_{\text{fin}} > 2$).

In [ ]:
# Dados
k_pin = 400.0
D_pin = 0.005
L_pin = 0.025
h_pin = 80.0

# Geometria
Ac_pin = np.pi * D_pin**2 / 4
P_pin = np.pi * D_pin
Ab = Ac_pin  # area da base

# Parametro m
m_pin = np.sqrt(h_pin * P_pin / (k_pin * Ac_pin))
mL_pin = m_pin * L_pin

# Eficiencia
eta_pin = np.tanh(mL_pin) / mL_pin

# Area superficial da aleta
A_fin = P_pin * L_pin

# Efetividade
epsilon = eta_pin * A_fin / Ab

print('Aleta Pin de Cobre:')
print('  D = {:.0f} mm, L = {:.0f} mm'.format(D_pin*1000, L_pin*1000))
print('  m = {:.2f} m^-1'.format(m_pin))
print('  mL = {:.3f}'.format(mL_pin))
print('  eta = {:.3f} ({:.1f}%)'.format(eta_pin, eta_pin*100))
print('  A_fin/Ab = {:.1f}'.format(A_fin/Ab))
print('\nEfetividade:')
print('  epsilon = {:.1f}'.format(epsilon))
if epsilon > 2:
    print('  >>> Aleta JUSTIFICAVEL (epsilon > 2) <<<')
else:
    print('  >>> Aleta NAO justificavel <<<')

---

## 🔧 Seção 3: Aleta Anular

### Exercício 3.1: Aleta Anular de Alumínio

Calcule a eficiência de uma aleta anular com $r_1 = 12,5$ mm, $r_2 = 25$ mm, $t = 0,4$ mm, $k = 237$ W/(m·K), $h = 75$ W/(m²·K).

In [ ]:
# Dados
r1 = 0.0125
r2 = 0.025
t_an = 0.0004
k_an = 237.0
h_an = 75.0

# Geometria
L_an = r2 - r1
Lc_an = L_an + t_an/2
r2c = r1 + Lc_an

# Aproximacao via aleta reta equivalente
Am = 2 * np.pi * ((r1 + r2)/2) * t_an  # area media
Pm = 2 * np.pi * ((r1 + r2)/2)

m_an = np.sqrt(h_an * Pm / (k_an * Am))
mLc_an = m_an * Lc_an

eta_an = np.tanh(mLc_an) / mLc_an

print('Aleta Anular:')
print('  r1 = {:.1f} mm, r2 = {:.1f} mm'.format(r1*1000, r2*1000))
print('  t  = {:.1f} mm'.format(t_an*1000))
print('  Lc = {:.2f} mm'.format(Lc_an*1000))
print('  m  = {:.2f} m^-1'.format(m_an))
print('  mLc = {:.3f}'.format(mLc_an))
print('\nEficiencia (aproximacao):')
print('  eta = {:.3f} ({:.1f}%)'.format(eta_an, eta_an*100))

---

## 📊 Seção 4: Visualização Comparativa

Vamos visualizar como a eficiência varia com $mL_c$ para diferentes geometrias.

In [ ]:
mL_range = np.linspace(0.1, 3, 100)
eta_range = np.tanh(mL_range) / mL_range

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(mL_range, eta_range, 'b-', linewidth=3)
ax.axhline(y=0.9, color='g', linestyle='--', label='eta = 90% (limiar)')
ax.axhline(y=0.5, color='r', linestyle='--', label='eta = 50%')

# Pontos dos exercicios
ax.plot(mLc, eta, 'ko', markersize=10, label='Ex 10.1 (reta)')
ax.plot(mL_pin, eta_pin, 'rs', markersize=10, label='Ex 10.2 (pin)')
ax.plot(mLc_an, eta_an, 'g^', markersize=10, label='Ex 10.3 (anular)')

ax.set_xlabel('mLc (parametro adimensional)', fontsize=12)
ax.set_ylabel('Eficiencia da Aleta, eta', fontsize=12)
ax.set_title('Eficiencia de Aleta vs. Parametro mLc', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

print('Interpretacao:')
print('  - mLc pequeno (< 0.5): alta eficiencia (> 90%)')
print('  - mLc moderado (~ 1): eficiencia moderada (~ 76%)')
print('  - mLc grande (> 2): baixa eficiencia (< 48%)')
print('\nPara alta eficiencia: use aletas curtas, finas e de alta condutividade.')

---

## 🔬 Estudos de Caso

Os estudos de caso aplicam os conceitos deste capítulo em problemas reais de engenharia.

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 10.1** | Condensador a Ar (ACC) com Tubos Aletados | [🔗 Abrir](../casos/10_1_Condensador_ACC_Tubos_Aletados.ipynb) |
| **Caso 10.2** | Dissipador Térmico para Processador | [🔗 Abrir](../casos/10_2_Dissipador_Termico_Processador.ipynb) |
| **Caso 10.3** | Aleta por Diferenças Finitas 1D | [🔗 Abrir](../casos/10_3_Aleta_Diferencas_Finitas_1D.ipynb) |

---

## 📖 Referências

- INCROPERA, F. P. et al. *Fundamentals of Heat and Mass Transfer*. 8ª ed. Wiley, 2021.
- KRAUS, A. D.; AZIZ, A.; WELTY, J. *Extended Surface Heat Transfer*. Wiley, 2001.
- CENGEL, Y. A. *Heat Transfer: A Practical Approach*. 4ª ed. McGraw-Hill, 2014.

---

## 🔄 Navegação

- [📚 Capítulo Anterior: Trocadores de Calor](09_Trocadores_Calor.ipynb)
- [📚 Próximo Capítulo: Advecção e Dispersão](11_Adveccao_Dispersao.ipynb)
- [📂 Outros Capítulos](../notebooks/)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">

**QR Code do Capítulo 10**  
Aponte seu celular para o QR Code no livro para acessar este notebook!

</div>

In [ ]:
print('=' * 60)
print('✅ NOTEBOOK CONCLUÍDO COM SUCESSO!')
print('=' * 60)
print('\n🎓 Você completou o Capítulo 10!')
print('📖 Próximo passo: Capítulo 11 - Advecção e Dispersão')
print('\n💡 Dica: Execute este notebook quantas vezes precisar!')
print('   Modifique os parâmetros e explore diferentes cenários.')